# Gamma Scalping — Gamma Sigma Prop Fund

## Strategy Overview

**Bond $100,000 USD** to EthStrategy protocol. Receive:

| Instrument | Detail | USD Value |
|---|---|---|
| 4yr American Call on ETH | 50 ETH notional, 20% OTM strike ($2,400) | received for free |
| 100,000 CDT | @ $0.50/CDT | $50,000 |

**Use the 50,000 USD (CDT collateral) to run a delta-neutral gamma scalp on the long call.**

### Why this works
- Normal gamma scalping profits when **Realised Vol > Implied Vol paid**.
- Here the call cost **$0** (received as bonding reward), so IV paid = 0.
- Therefore **all realised volatility is pure profit** — we are playing with house money.
- The hedge is an ETH short (delta hedge). In crypto, shorts receive funding — treated as negligible here.
- The only real risk: collateral (50k CDT) liquidated if ETH rips violently against the short.

### Rebalance cadence
Daily — recalculate delta, adjust short to maintain delta neutrality.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import norm

plt.rcParams.update({
    'figure.facecolor': '#F0EBE0',
    'axes.facecolor':   '#E8E1D3',
    'axes.edgecolor':   '#CFC6B4',
    'axes.labelcolor':  '#0E0C0A',
    'xtick.color':      '#3D3830',
    'ytick.color':      '#3D3830',
    'text.color':       '#0E0C0A',
    'grid.color':       '#CFC6B4',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
})

## 1. Protocol & Position Parameters

In [ ]:
# ── Bond parameters ──────────────────────────────────────────────
BOND_AMOUNT       = 100_000.0   # USD lent to protocol
CDT_RECEIVED      = 100_000     # CDT tokens received
CDT_PRICE         = 0.50        # USD per CDT
COLLATERAL_USD    = CDT_RECEIVED * CDT_PRICE   # 50,000 USD collateral for hedge

# ── ETH call option received ─────────────────────────────────────
ETH_SPOT          = 2_000.0     # ETH price at bond entry
N_ETH             = 50.0        # notional ETH (100k bond / 2k ETH)
STRIKE_PCT        = 1.20        # 20% OTM
STRIKE            = ETH_SPOT * STRIKE_PCT          # $2,400
T_YEARS           = 4.0         # option tenor
RISK_FREE         = 0.05        # annualised risk-free rate
IV                = 0.75        # market IV for 4yr ETH call (~75% long-dated ETH vol)
COST_BASIS        = 0.0         # option received for free — IV paid = 0

# ── Scalping parameters ──────────────────────────────────────────
REBALANCE_DAYS    = 1           # daily rebalance
SIM_DAYS          = int(T_YEARS * 365)   # full 4-year life of the option
RV                = 0.80        # assumed realised vol over simulation period
SEED              = 42

print('=== EthStrategy Bond — Position Summary ===')
print(f'  Bond amount       : ${BOND_AMOUNT:>10,.2f}')
print(f'  CDT received      : {CDT_RECEIVED:>10,} CDT  (${COLLATERAL_USD:,.2f})')
print(f'  Collateral (hedge): ${COLLATERAL_USD:>10,.2f}')
print()
print(f'  ETH spot          : ${ETH_SPOT:>10,.2f}')
print(f'  Option notional   : {N_ETH:.0f} ETH')
print(f'  Strike            : ${STRIKE:>10,.2f}  ({STRIKE_PCT:.0%} OTM)')
print(f'  Tenor             : {T_YEARS:.0f} years')
print(f'  Market IV         : {IV:.0%}')
print(f'  Cost basis        : ${COST_BASIS}  (free — IV paid = 0%)')
print(f'  Breakeven RV      : 0%  (all vol is profit)')
print(f'  Simulation period : {SIM_DAYS} days ({T_YEARS:.0f} years — full option life)')

## 2. Black-Scholes Greeks

For a European call on a non-dividend paying asset, the American call price equals the European call price — early exercise is never optimal. We use Black-Scholes directly.

In [ ]:
def bs_greeks(S, K, T, r, sigma):
    """Black-Scholes Greeks for a long call. Returns price, delta, gamma, theta, vega."""
    if T <= 1e-8:
        return {'price': max(S - K, 0), 'delta': 1.0 if S > K else 0.0,
                'gamma': 0.0, 'theta': 0.0, 'vega': 0.0}
    d1    = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2    = d1 - sigma * np.sqrt(T)
    pdf1  = norm.pdf(d1)
    price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = pdf1 / (S * sigma * np.sqrt(T))
    # theta: daily (divide annualised by 365)
    theta = (-(S * pdf1 * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    vega  = S * pdf1 * np.sqrt(T) / 100   # per 1 vol point
    return {'price': price, 'delta': delta, 'gamma': gamma, 'theta': theta, 'vega': vega}


# Greeks at entry
g0 = bs_greeks(ETH_SPOT, STRIKE, T_YEARS, RISK_FREE, IV)

print('=== Call Greeks at Entry ===')
print(f'  Option price (BS) : ${g0["price"]:>8.2f} per ETH  |  ${g0["price"] * N_ETH:,.2f} total')
print(f'  Delta             : {g0["delta"]:>8.4f}  →  short {g0["delta"] * N_ETH:.2f} ETH to hedge')
print(f'  Gamma             : {g0["gamma"]:>8.6f}  per ETH  |  {g0["gamma"] * N_ETH:.6f} total')
print(f'  Theta (daily)     : ${g0["theta"]:>8.4f} per ETH  |  ${g0["theta"] * N_ETH:,.2f}/day total')
print(f'  Vega              : ${g0["vega"]:>8.4f} per ETH per vol pt')
print()
print(f'  Theoretical value received: ${g0["price"] * N_ETH:,.2f}  (cost basis: $0)')
print(f'  Free edge        : ${g0["price"] * N_ETH:,.2f}')

## 3. Collateral & Margin Check

The short ETH position is collateralised by the 50k USD (CDT). Check that the initial hedge fits within the collateral.

In [ ]:
initial_short_eth  = g0['delta'] * N_ETH          # ETH to short
initial_short_usd  = initial_short_eth * ETH_SPOT  # notional value of short
collateral_ratio   = COLLATERAL_USD / initial_short_usd

# Liquidation estimate: short gets liquidated if ETH rises enough to wipe collateral
# Loss on short = short_eth * (S_liq - S0)
# => S_liq = S0 + COLLATERAL_USD / short_eth
liq_price = ETH_SPOT + COLLATERAL_USD / initial_short_eth
liq_pct   = (liq_price / ETH_SPOT - 1)

print('=== Collateral & Margin ===')
print(f'  Collateral available : ${COLLATERAL_USD:,.2f}')
print(f'  Initial short        : {initial_short_eth:.2f} ETH  (${initial_short_usd:,.2f} notional)')
print(f'  Collateral ratio     : {collateral_ratio:.2f}x')
print()
print(f'  Liquidation price    : ~${liq_price:,.2f}  ({liq_pct:+.1%} from spot)')
print(f'  Note: call delta increases as ETH rises, partially offsetting short losses.')
print(f'  Net delta stays near 0 after each daily rebalance — liquidation risk is managed.')

## 4. Daily Gamma Scalp Simulation

In [ ]:
np.random.seed(SEED)
dt = 1 / 365

# Simulate ETH price path
log_returns = (RISK_FREE - 0.5 * RV**2) * dt + RV * np.sqrt(dt) * np.random.randn(SIM_DAYS)
prices      = ETH_SPOT * np.cumprod(np.insert(np.exp(log_returns), 0, 1.0))

# State
short_eth      = initial_short_eth   # current ETH short position
short_entry_px = ETH_SPOT
collateral     = COLLATERAL_USD
realised_pnl   = 0.0
records        = []

for day in range(len(prices)):
    S      = prices[day]
    T_rem  = max(T_YEARS - day * dt, 1e-8)
    g      = bs_greeks(S, STRIKE, T_rem, RISK_FREE, IV)

    # P&L on existing short since last rebalance
    short_pnl_today = short_eth * (short_entry_px - S)

    # Option mark-to-market
    option_mtm = g['price'] * N_ETH

    # Net delta before rebalance
    call_delta_total = g['delta'] * N_ETH
    net_delta        = call_delta_total - short_eth

    # Daily gamma scalp P&L approximation: ½ * Γ_total * (ΔS)²
    dS = S - (prices[day - 1] if day > 0 else ETH_SPOT)
    gamma_pnl_approx = 0.5 * g['gamma'] * N_ETH * dS**2
    theta_cost       = g['theta'] * N_ETH

    # Realise short P&L and rebalance
    realised_pnl  += short_pnl_today
    short_eth      = call_delta_total
    short_entry_px = S

    # Update collateral
    collateral = COLLATERAL_USD + realised_pnl

    # ── Dynamic liquidation price ─────────────────────────────────
    # After rebalancing, short_eth is the new position.
    # Liquidation when: collateral + short_eth * (S - S_liq) = 0
    # => S_liq = S + collateral / short_eth
    # Distance = S_liq - S = collateral / short_eth
    if short_eth > 0:
        liq_price      = S + collateral / short_eth
        distance_to_liq = liq_price - S
        distance_pct    = distance_to_liq / S
    else:
        liq_price       = np.nan
        distance_to_liq = np.nan
        distance_pct    = np.nan

    records.append({
        'day':              day,
        'ETH':              S,
        'T_rem_years':      T_rem,
        'call_delta':       g['delta'],
        'call_gamma':       g['gamma'],
        'call_theta':       g['theta'],
        'short_eth':        short_eth,
        'net_delta':        net_delta,
        'option_mtm':       option_mtm,
        'realised_pnl':     realised_pnl,
        'gamma_pnl_approx': gamma_pnl_approx,
        'theta_cost':       theta_cost,
        'collateral':       collateral,
        'liq_price':        liq_price,
        'distance_to_liq':  distance_to_liq,
        'distance_pct':     distance_pct,
    })

df = pd.DataFrame(records)

# ── Margin summary ────────────────────────────────────────────────
min_dist_row = df.loc[df['distance_to_liq'].idxmin()]
print(f'=== Simulation Results ({SIM_DAYS} days, RV={RV:.0%}) ===')
print(f'  ETH start              : ${ETH_SPOT:,.2f}')
print(f'  ETH end                : ${df["ETH"].iloc[-1]:,.2f}')
print(f'  Realised scalp P&L     : ${df["realised_pnl"].iloc[-1]:,.2f}')
print(f'  Option MTM (end)       : ${df["option_mtm"].iloc[-1]:,.2f}  (cost basis $0)')
print(f'  Total P&L              : ${df["realised_pnl"].iloc[-1] + df["option_mtm"].iloc[-1]:,.2f}')
print(f'  Collateral (end)       : ${df["collateral"].iloc[-1]:,.2f}')
print()
print(f'=== Margin Health ===')
print(f'  Entry liq price        : ${df["liq_price"].iloc[0]:,.2f}  ({df["distance_pct"].iloc[0]:+.1%} from entry)')
print(f'  Final liq price        : ${df["liq_price"].iloc[-1]:,.2f}  ({df["distance_pct"].iloc[-1]:+.1%} from spot)')
print(f'  Closest to liquidation : day {int(min_dist_row["day"])}  |  '
      f'ETH=${min_dist_row["ETH"]:,.2f}  |  liq=${min_dist_row["liq_price"]:,.2f}  '
      f'({min_dist_row["distance_pct"]:+.1%} buffer)')
print(f'  Min collateral         : ${df["collateral"].min():,.2f}  (day {df["collateral"].idxmin()})')
liquidated = (df['collateral'] <= 0).any()
print(f'  Liquidated             : {"YES — position blown" if liquidated else "No"}')

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 16), sharex=True)
fig.suptitle(
    f'EthStrategy Gamma Scalp  |  50 ETH Call (4yr, 20% OTM)  |  RV {RV:.0%}  vs  IV {IV:.0%}',
    fontsize=13, fontweight='bold', y=0.99
)

days = df['day']

# Panel 1 — ETH price path + dynamic liquidation level
ax = axes[0]
ax.plot(days, df['ETH'],       color='#4A6FA5', lw=1.5, label='ETH Price')
ax.plot(days, df['liq_price'], color='#C0392B', lw=1.2, ls='--', label='Liq. Price (dynamic)')
ax.axhline(STRIKE,   color='#E67E22', lw=0.8, ls=':', label=f'Strike ${STRIKE:,.0f}')
ax.axhline(ETH_SPOT, color='#7A7168', lw=0.7, ls=':')
ax.fill_between(days, df['ETH'], df['liq_price'],
                where=(df['liq_price'] > df['ETH']),
                alpha=0.08, color='#C0392B', label='Buffer zone')
ax.set_ylabel('ETH Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)

# Panel 2 — Distance to liquidation (% buffer)
ax = axes[1]
ax.plot(days, df['distance_pct'] * 100, color='#C0392B', lw=1.3)
ax.fill_between(days, df['distance_pct'] * 100, 0,
                where=(df['distance_pct'] > 0), alpha=0.15, color='#27AE60')
ax.fill_between(days, df['distance_pct'] * 100, 0,
                where=(df['distance_pct'] <= 0), alpha=0.3, color='#C0392B')
ax.axhline(0,  color='#C0392B', lw=1.0, ls='--', label='Liquidation (0%)')
ax.axhline(20, color='#E67E22', lw=0.8, ls=':', label='Warning threshold (20%)')
ax.set_ylabel('Buffer to Liq. (%)')
ax.legend(fontsize=8)
ax.grid(True)

# Panel 3 — Realised scalp P&L + option MTM
ax = axes[2]
ax.plot(days, df['realised_pnl'], color='#27AE60', lw=1.5, label='Realised Scalp P&L')
ax.plot(days, df['option_mtm'],   color='#4A6FA5', lw=1.2, ls='--', label='Option MTM (cost basis $0)')
ax.axhline(0, color='#7A7168', lw=0.7)
ax.set_ylabel('P&L ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)

# Panel 4 — Collateral balance
ax = axes[3]
ax.plot(days, df['collateral'], color='#E67E22', lw=1.5, label='Collateral (USD)')
ax.axhline(0,               color='#C0392B', lw=1.0, ls='--', label='Liquidation')
ax.axhline(COLLATERAL_USD,  color='#7A7168', lw=0.7, ls=':',  label=f'Initial ${COLLATERAL_USD:,.0f}')
ax.fill_between(days, df['collateral'], 0, where=(df['collateral'] > 0),
                alpha=0.15, color='#27AE60')
ax.fill_between(days, df['collateral'], 0, where=(df['collateral'] < 0),
                alpha=0.3, color='#C0392B')
ax.set_ylabel('Collateral ($)')
ax.set_xlabel('Days')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.savefig('gamma_scalp_ethstrategy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['cum_gamma_pnl'] = df['gamma_pnl_approx'].cumsum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('Gamma Scalp — Pure P&L vs Hedge Size', fontsize=13, fontweight='bold')

# Panel 1 — Cumulative gamma scalp profits
ax1.plot(df['day'], df['cum_gamma_pnl'], color='#27AE60', lw=1.8, label='Cumulative gamma P&L')
ax1.bar(df['day'], df['gamma_pnl_approx'], color='#27AE60', alpha=0.25, width=1, label='Daily ½Γ(ΔS)²')
ax1.axhline(0, color='#7A7168', lw=0.7)
ax1.set_ylabel('P&L ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(fontsize=8)
ax1.grid(True)

# Panel 2 — Hedge size (ETH short notional in USD)
hedge_notional = df['short_eth'] * df['ETH']
ax2.plot(df['day'], hedge_notional, color='#C0392B', lw=1.5, label='Short notional (USD)')
ax2.fill_between(df['day'], hedge_notional, alpha=0.1, color='#C0392B')
ax2_r = ax2.twinx()
ax2_r.plot(df['day'], df['short_eth'], color='#8E44AD', lw=1.2, ls='--', label='Short size (ETH)')
ax2_r.set_ylabel('ETH Short', color='#8E44AD')
ax2_r.tick_params(axis='y', labelcolor='#8E44AD')
ax2.set_ylabel('Short Notional ($)')
ax2.set_xlabel('Days')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.legend(fontsize=8, loc='upper left')
ax2_r.legend(fontsize=8, loc='upper right')
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'Total gamma scalp P&L : ${df["cum_gamma_pnl"].iloc[-1]:,.2f}')
print(f'Avg daily P&L         : ${df["gamma_pnl_approx"].mean():,.2f}')
print(f'Avg short size        : {df["short_eth"].mean():.2f} ETH  (${(df["short_eth"] * df["ETH"]).mean():,.2f} notional)')

In [ ]:
# Gamma of the 50 ETH position over the full 4-year life
# Shown at several ETH price levels — illustrates how gamma explodes as ETH approaches strike

t_range    = np.linspace(0.01, T_YEARS - 0.01, 500)
eth_levels = [1600, 1800, 2000, 2200, 2400, 2600]
colors     = ['#7A7168', '#A89F94', '#4A6FA5', '#8E44AD', '#E67E22', '#27AE60']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
fig.suptitle('Option Gamma Over 4-Year Life  |  Strike $2,400  |  50 ETH Position',
             fontsize=13, fontweight='bold')

for eth, col in zip(eth_levels, colors):
    gammas    = [bs_greeks(eth, STRIKE, T_YEARS - t, RISK_FREE, IV)['gamma'] * N_ETH
                 for t in t_range]
    moneyness = f'ATM' if eth == STRIKE else f'{(eth/STRIKE - 1):+.0%}'
    ax1.plot(t_range, gammas, color=col, lw=1.6, label=f'ETH ${eth:,}  ({moneyness})')

for yr in [1, 2, 3]:
    ax1.axvline(yr, color='#CFC6B4', lw=0.8, ls=':')
ax1.set_ylabel('Total Gamma')
ax1.legend(fontsize=8)
ax1.grid(True)

# Panel 2 — Expected daily scalp P&L (½ × Γ × S² × RV² / 365)
for eth, col in zip(eth_levels, colors):
    daily_pnl = [0.5 * bs_greeks(eth, STRIKE, T_YEARS - t, RISK_FREE, IV)['gamma']
                 * N_ETH * eth**2 * RV**2 / 365
                 for t in t_range]
    ax2.plot(t_range, daily_pnl, color=col, lw=1.6)

for yr in [1, 2, 3]:
    ax2.axvline(yr, color='#CFC6B4', lw=0.8, ls=':')
ax2.set_ylabel('Expected Daily Scalp P&L ($)')
ax2.set_xlabel('Years Elapsed')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.1f}'))
ax2.grid(True)

plt.tight_layout()
plt.show()

# Table: gamma and expected daily P&L at ETH=$2,000 over time
print('Gamma & expected daily scalp P&L at ETH=$2,000:')
print(f'  {"Year":<8} {"Gamma (total)":>14} {"Daily P&L":>12} {"Ann. P&L":>12}')
for yr in [0, 0.5, 1, 2, 3, 3.9]:
    T_rem = T_YEARS - max(yr, 0.01)
    g_val = bs_greeks(ETH_SPOT, STRIKE, T_rem, RISK_FREE, IV)['gamma'] * N_ETH
    dpnl  = 0.5 * g_val * ETH_SPOT**2 * RV**2 / 365
    print(f'  {yr:<8.1f} {g_val:>14.6f} {dpnl:>12.2f} {dpnl*365:>12.2f}')

In [ ]:
# ── Monte Carlo: 200 simulated 4-year gamma scalps ───────────────────────────
# Rebalance only when delta drifts beyond REBALANCE_BAND from current hedge.
# On trending paths this slows short growth, reducing bleed.
# On oscillating paths most of the gamma scalp is still captured.
N_SIMS          = 200
EXERCISE_ITM    = 0.20   # exercise when 20% ITM
FUNDING_RATE    = 0.10   # annualised funding earned by short
REBALANCE_BAND  = 0.10   # only rebalance when |current_delta - hedge_delta| >= 0.10

mc_results = []

for sim in range(N_SIMS):
    np.random.seed(sim)
    log_returns = (RISK_FREE - 0.5 * RV**2) * dt + RV * np.sqrt(dt) * np.random.randn(SIM_DAYS)
    prices_mc   = ETH_SPOT * np.cumprod(np.insert(np.exp(log_returns), 0, 1.0))

    short_eth_mc      = g0['delta'] * N_ETH
    short_entry_px_mc = ETH_SPOT
    collateral_mc     = COLLATERAL_USD
    realised_pnl_mc   = 0.0
    funding_pnl_mc    = 0.0
    liquidated_mc     = False
    exercised_mc      = False
    liq_day_mc        = None
    exercise_day_mc   = None
    exercise_price_mc = None
    min_collateral_mc = collateral_mc
    eth_path          = []
    collateral_path   = []
    pnl_path          = []

    for day in range(len(prices_mc)):
        S     = prices_mc[day]
        T_rem = max(T_YEARS - day * dt, 1e-8)

        if exercised_mc or liquidated_mc:
            eth_path.append(S)
            collateral_path.append(collateral_mc)
            pnl_path.append(realised_pnl_mc)
            continue

        g = bs_greeks(S, STRIKE, T_rem, RISK_FREE, IV)

        # Funding on existing short
        funding_today    = short_eth_mc * S * (FUNDING_RATE / 365)
        funding_pnl_mc  += funding_today

        # P&L on existing short
        short_pnl_today  = short_eth_mc * (short_entry_px_mc - S)
        realised_pnl_mc += short_pnl_today + funding_today
        collateral_mc    = COLLATERAL_USD + realised_pnl_mc

        # Always roll entry price forward
        short_entry_px_mc = S

        # Liquidation
        if collateral_mc <= 0:
            realised_pnl_mc = -COLLATERAL_USD
            collateral_mc   = 0.0
            liquidated_mc   = True
            liq_day_mc      = day
            short_eth_mc    = 0.0
            eth_path.append(S)
            collateral_path.append(collateral_mc)
            pnl_path.append(realised_pnl_mc)
            continue

        # Early exercise: 20% ITM
        if S >= STRIKE * (1 + EXERCISE_ITM):
            intrinsic         = (S - STRIKE) * N_ETH
            realised_pnl_mc  += intrinsic
            collateral_mc     = COLLATERAL_USD + realised_pnl_mc
            exercised_mc      = True
            exercise_day_mc   = day
            exercise_price_mc = S
            short_eth_mc      = 0.0
            eth_path.append(S)
            collateral_path.append(collateral_mc)
            pnl_path.append(realised_pnl_mc)
            continue

        # Band rebalance: only adjust short if delta has drifted >= REBALANCE_BAND
        current_hedge_delta = short_eth_mc / N_ETH
        if abs(g['delta'] - current_hedge_delta) >= REBALANCE_BAND:
            short_eth_mc = g['delta'] * N_ETH

        min_collateral_mc = min(min_collateral_mc, collateral_mc)
        eth_path.append(S)
        collateral_path.append(collateral_mc)
        pnl_path.append(realised_pnl_mc)

    # Collect intrinsic at expiry if not already exercised
    if not exercised_mc:
        realised_pnl_mc += max(prices_mc[-1] - STRIKE, 0) * N_ETH

    total_pnl = realised_pnl_mc

    mc_results.append({
        'sim':              sim,
        'eth_final':        prices_mc[-1],
        'realised_pnl':     realised_pnl_mc,
        'funding_pnl':      funding_pnl_mc,
        'total_pnl':        total_pnl,
        'min_collateral':   min_collateral_mc,
        'liquidated':       liquidated_mc,
        'liq_day':          liq_day_mc,
        'exercised':        exercised_mc,
        'exercise_day':     exercise_day_mc,
        'exercise_price':   exercise_price_mc,
        'eth_path':         eth_path,
        'collateral_path':  collateral_path,
        'pnl_path':         pnl_path,
    })

mc = pd.DataFrame([{k: v for k, v in r.items()
                     if k not in ('eth_path', 'collateral_path', 'pnl_path')}
                    for r in mc_results])

liq_rate      = mc['liquidated'].mean()
exercise_rate = mc['exercised'].mean()
neg_rate      = (mc['total_pnl'] < 0).mean()
print(f'=== Monte Carlo Results  ({N_SIMS} sims, RV={RV:.0%}, band=±{REBALANCE_BAND}) ===')
print(f'  Exercise rule          : {EXERCISE_ITM:.0%} ITM')
print(f'  Liquidation rate       : {liq_rate:.1%}  ({int(mc["liquidated"].sum())} / {N_SIMS} sims)')
print(f'  Early exercise rate    : {exercise_rate:.1%}  ({int(mc["exercised"].sum())} / {N_SIMS} sims)')
print(f'  Negative P&L rate      : {neg_rate:.1%}  ({int((mc["total_pnl"] < 0).sum())} / {N_SIMS} sims)')
print(f'  Ran to expiry          : {int((~mc["liquidated"] & ~mc["exercised"]).sum())} sims')
print()
print(f'  Median total P&L       : ${mc["total_pnl"].median():>10,.0f}')
print(f'  Mean total P&L         : ${mc["total_pnl"].mean():>10,.0f}')
print(f'  5th pct total P&L      : ${mc["total_pnl"].quantile(0.05):>10,.0f}')
print(f'  95th pct total P&L     : ${mc["total_pnl"].quantile(0.95):>10,.0f}')
print(f'  Median funding earned  : ${mc["funding_pnl"].median():>10,.0f}')

# ── Plots ─────────────────────────────────────────────────────────────────────
n_path    = len(mc_results[0]['eth_path'])
days_axis = np.arange(n_path)

fig, axes = plt.subplots(4, 1, figsize=(13, 18), sharex=False)
fig.suptitle(f'Monte Carlo Gamma Scalp  |  {N_SIMS} Sims  |  RV={RV:.0%}  IV={IV:.0%}  '
             f'|  Exercise @ {EXERCISE_ITM:.0%} ITM  |  Band ±{REBALANCE_BAND}  |  Funding={FUNDING_RATE:.0%}/yr',
             fontsize=12, fontweight='bold')

ax = axes[0]
eth_paths_arr = np.array([r['eth_path'] for r in mc_results])
for r, row in zip(mc_results, mc.itertuples()):
    col = '#C0392B' if row.liquidated else ('#E67E22' if row.exercised else '#4A6FA5')
    ax.plot(days_axis, r['eth_path'], lw=0.4, alpha=0.15, color=col)
ax.plot(days_axis, np.percentile(eth_paths_arr, 5,  axis=0), color='#7A7168', lw=1.2, ls='--', label='5th/95th pct')
ax.plot(days_axis, np.percentile(eth_paths_arr, 95, axis=0), color='#7A7168', lw=1.2, ls='--')
ax.plot(days_axis, np.median(eth_paths_arr, axis=0),          color='#4A6FA5', lw=1.8, label='Median')
ax.axhline(STRIKE,                    color='#E67E22', lw=1.0, ls=':', label=f'Strike ${STRIKE:,.0f}')
ax.axhline(STRIKE * (1+EXERCISE_ITM), color='#27AE60', lw=1.0, ls=':', label=f'Exercise trigger ${STRIKE*(1+EXERCISE_ITM):,.0f}')
ax.set_ylabel('ETH Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)
ax.set_title('ETH Paths  (blue=full term, orange=exercised early, red=liquidated)')

ax = axes[1]
coll_arr = np.array([r['collateral_path'] for r in mc_results])
for r, row in zip(mc_results, mc.itertuples()):
    col = '#C0392B' if row.liquidated else ('#E67E22' if row.exercised else '#27AE60')
    ax.plot(days_axis, r['collateral_path'], lw=0.4, alpha=0.12, color=col)
ax.plot(days_axis, np.percentile(coll_arr, 5,  axis=0), color='#C0392B', lw=1.2, ls='--', label='5th pct')
ax.plot(days_axis, np.percentile(coll_arr, 50, axis=0), color='#27AE60', lw=1.8,           label='Median')
ax.axhline(0,              color='#C0392B', lw=1.2, ls='--', label='Liquidation')
ax.axhline(COLLATERAL_USD, color='#7A7168', lw=0.8, ls=':',  label=f'Initial ${COLLATERAL_USD:,.0f}')
ax.set_ylabel('Collateral / Cash ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)
ax.set_title(f'Collateral Health  |  Liq: {liq_rate:.1%}  |  Exercised: {exercise_rate:.1%}')

ax = axes[2]
pnl_arr = np.array([r['pnl_path'] for r in mc_results])
for r in mc_results:
    ax.plot(days_axis, r['pnl_path'], lw=0.4, alpha=0.12, color='#27AE60')
ax.plot(days_axis, np.percentile(pnl_arr, 5,  axis=0), color='#C0392B', lw=1.2, ls='--', label='5th pct')
ax.plot(days_axis, np.percentile(pnl_arr, 95, axis=0), color='#27AE60', lw=1.2, ls='--', label='95th pct')
ax.plot(days_axis, np.median(pnl_arr, axis=0),          color='#27AE60', lw=1.8,           label='Median')
ax.axhline(0, color='#7A7168', lw=0.7)
ax.set_ylabel('Total P&L ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)
ax.set_title('Cumulative P&L (scalp + funding + intrinsic on exercise/expiry)')

ax = axes[3]
liq_mask  = mc['liquidated']
ex_mask   = mc['exercised'] & ~liq_mask
full_mask = ~liq_mask & ~ex_mask
ax.hist(mc[full_mask]['total_pnl'], bins=30, color='#4A6FA5', alpha=0.7, label=f'Full term ({full_mask.sum()})')
ax.hist(mc[ex_mask]['total_pnl'],   bins=30, color='#E67E22', alpha=0.7, label=f'Exercised early ({ex_mask.sum()})')
ax.hist(mc[liq_mask]['total_pnl'],  bins=20, color='#C0392B', alpha=0.7, label=f'Liquidated ({liq_mask.sum()})')
ax.axvline(mc['total_pnl'].median(), color='black', lw=1.5, ls='--',
           label=f'Median ${mc["total_pnl"].median():,.0f}')
ax.axvline(0, color='#7A7168', lw=0.8, ls=':')
ax.set_xlabel('Total P&L ($)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8)
ax.grid(True)
ax.set_title('P&L Distribution by Outcome')

plt.tight_layout()
plt.savefig('gamma_scalp_montecarlo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── When does liquidation occur? ─────────────────────────────────────────────
liq_sims = mc[mc['liquidated']].copy()
liq_sims['liq_year'] = liq_sims['liq_day'] / 365

print(f'=== Liquidation Timing  ({len(liq_sims)} liquidated sims) ===')
print(f'  Median liq day   : day {liq_sims["liq_day"].median():.0f}  (yr {liq_sims["liq_year"].median():.2f})')
print(f'  Earliest liq     : day {liq_sims["liq_day"].min():.0f}  (yr {liq_sims["liq_year"].min():.2f})')
print(f'  Latest liq       : day {liq_sims["liq_day"].max():.0f}  (yr {liq_sims["liq_year"].max():.2f})')
print()
for yr_cut in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    n = (liq_sims['liq_year'] <= yr_cut).sum()
    print(f'  Liquidated by yr {yr_cut:.1f} : {n:>3}  ({n/len(liq_sims):.0%} of liq sims, '
          f'{n/N_SIMS:.0%} of all sims)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Liquidation Timing', fontsize=12, fontweight='bold')

# Histogram of liquidation day
ax = axes[0]
ax.hist(liq_sims['liq_year'], bins=30, color='#C0392B', alpha=0.75, edgecolor='white')
ax.axvline(liq_sims['liq_year'].median(), color='#4A6FA5', lw=1.5, ls='--',
           label=f'Median yr {liq_sims["liq_year"].median():.2f}')
for yr in [1, 2, 3]:
    ax.axvline(yr, color='#CFC6B4', lw=0.8, ls=':')
ax.set_xlabel('Year of Liquidation')
ax.set_ylabel('Count')
ax.set_title('When do liquidations happen?')
ax.legend(fontsize=8)
ax.grid(True)

# Cumulative liquidation rate over time
ax = axes[1]
liq_days_sorted = np.sort(liq_sims['liq_day'].values)
cum_liq = np.arange(1, len(liq_days_sorted) + 1) / N_SIMS * 100
ax.plot(liq_days_sorted / 365, cum_liq, color='#C0392B', lw=2)
ax.fill_between(liq_days_sorted / 365, cum_liq, alpha=0.15, color='#C0392B')
for yr in [1, 2, 3]:
    ax.axvline(yr, color='#CFC6B4', lw=0.8, ls=':')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative liquidation rate (%)')
ax.set_title('Cumulative liquidation rate over time')
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. Annualised Yield on Bond

Express the scalp P&L as a yield on the $100k bonded amount.

---
## Notes & Next Steps

- **Collateral risk**: If ETH makes a large sustained move up before the daily rebalance, the short bleeds. The call gains in value (long delta from option increases), but unrealised option gains don't protect the short margin intraday. Consider a wider rebalance band or stop-loss on the short.
- **CDT price risk**: Collateral is denominated in CDT. If CDT depegs below $0.50, effective collateral shrinks — model this separately.
- **Funding rate**: Set to zero per assumption. If ETH shorts pay positive funding (longs pay shorts), this adds incremental yield.
- **American vs European**: For a non-dividend paying asset, the American call = European call. If ETH staking yield is modelled as a continuous dividend `q`, early exercise may become optimal — extend `bs_greeks` to `bs_greeks_div(S, K, T, r, q, sigma)` in that case.